In [1]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,549 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,300 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packag

In [2]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 28, 2026 3:02:43 PM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->  void runPy(String code) throws Exception {
>>          String marker = "___END_OF_PYTHON_CELL___";
>>     
>>         // Base64 encode code payload to bypass all quote & newline syntax errors
>>          String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));
>>     
>>          // Guarantee marker printing using try...finally and display trailing expressions using ast
>>         String payload = String.format(
>>              "import base64, ast, traceback\n" +
>>             "try:\n" +
>>   

In [3]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_7'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Cloning into '/content/BITS_programming'...
Updating files:  11% (163/1417)
Updating files:  12% (171/1417)
Updating files:  13% (185/1417)
Updating files:  14% (199/1417)
Updating files:  15% (213/1417)
Updating files:  16% (227/1417)
Updating files:  17% (241/1417)
Updating files:  18% (256/1417)
Updating files:  19% (270/1417)
Updating files:  20% (284/1417)
Updating files:  21% (298/1417)
Updating files:  22% (312/1417)
Updating files:  23% (326/1417)
Updating files:  24% (341/1417)
Updating files:  25% (355/1417)
Updating files:  26% (369/1417)
Updating files:  27% (383/1417)
Updating files:  28% (397/1417)
Updating files:  29% (411/1417)
Updating files:  30% (426/1417)
Updating files:  31% (440/1417)
Updating files:  32% (454/1417)
Updating files:  33% (468/1417)
Updating files

In [4]:
%%jshell

runPy("""
# Import required libraries
import numpy as np

from sklearn.decomposition import PCA # Class that performs PCA
from sklearn.datasets import load_breast_cancer # To load the dataset on the fly
from sklearn.preprocessing import StandardScaler # To standardize the data

import matplotlib.pyplot as plt # Visualization
""");


In [5]:
%%jshell

runPy("""
# Dimensionality Reduction using Principal Component Analysis (PCA)
#
# In this lab, we perform Principal Component Analysis on the breast cancer
# dataset.
#
# PCA is a technique used to reduce the dimensionality of a dataset while
# retaining as much of the original variance as possible.
#
# This can be useful for visualizing high-dimensional data or for improving the
# performance of machine learning algorithms by removing noise and
# multicollinearity.
#
# We have primarily used it here to visualize the
""");


In [6]:
%%jshell

runPy("""
# Load the Breast Cancer dataset
dataset = load_breast_cancer()
dataset.data
dataset.target
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 0 0 1 0 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 0
 1 0 1 0 0 1 1 1 0 0 1 0 0 0 1 1 1 0 1 1 0 0 1 1 1 0 0 1 1 1 1 0 1 1 0 1 1
 1 1 1 1 1 1 0 0 0 1 0 0 1 1 1 0 0 1 0 1 0 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0 1
 1 1 1 1 1 1 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 1 1 0 0 1 1 1 1 0 1 1 0 0 0 1 0
 1 0 1 1 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 1 1 0 1 0 0 0 0 1 1 0 0 1 1
 1 0 1 1 1 1 1 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 1 1 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 1 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 0 1 1 1 1 0 0 0 1 1
 1 1 0 1 0 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0
 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 1 0 1 1 1 1 1 0 1 1
 0 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1

## Dataset Description and PCA's Role

This dataset contains information about breast cell nuclei, with each sample described by 30 measurements (features). These features include things like the size, shape, and texture of the cell nuclei. The goal is to determine if a mass is malignant (cancerous) or benign (not cancerous) based on these features.

Having 30 features makes it hard to visualize the data directly. Principal Component Analysis (PCA) helps by reducing these 30 features to a smaller number, in this case, two main components. PCA finds the directions in the data where there is the most variation.

By using PCA to reduce the data to 2 components, we can:

1.  **Simplify the data:** Make it easier to see patterns by going from 30 dimensions to just 2.
2.  **See the separation:** The plot shows how well the first two components separate the malignant and benign cases, indicating that PCA captures the key differences.
3.  **Reduce noise:** Focus on the most important variations in the data, ignoring less important ones.

PCA makes this high-dimensional breast cancer data easier to understand and visualize, especially for seeing the difference between cancerous and non-cancerous cells.

In [7]:
%%jshell

runPy("""
# Get the data from the dataset
from sklearn.datasets import load_breast_cancer # Ensure import for dataset loading
dataset = load_breast_cancer() # Added to ensure dataset is defined
data = dataset.data

# Display the (number of rows, dimensionality of each row)
data.shape
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... (569, 30)


In [8]:
%%jshell

runPy("""
# Standardize the data
#
# Standardizing the data is important for PCA because it is sensitive to the
# scale of the features.
#
# Print the scaled data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)
print(scaled_data)
scaled_data.shape
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... [[ 1.09706398 -2.07333501  1.26993369 ...  2.29607613  2.75062224
   1.93701461]
 [ 1.82982061 -0.35363241  1.68595471 ...  1.0870843  -0.24388967
   0.28118999]
 [ 1.57988811  0.45618695  1.56650313 ...  1.95500035  1.152255
   0.20139121]
 ...
 [ 0.70228425  2.0455738   0.67267578 ...  0.41406869 -1.10454895
  -0.31840916]
 [ 1.83834103  2.33645719  1.98252415 ...  2.28998549  1.91908301
   2.21963528]
 [-1.80840125  1.22179204 -1.81438851 ... -1.74506282 -0.04813821
  -0.75120669]]
(569, 30)


In [9]:
%%jshell

runPy("""
# Create a PCA instance with 2 components
#
# We choose 2 components to reduce the data to a 2-dimensional space,
#  which is useful for visualization.
n_components = 2
pca = PCA(n_components=n_components)
""");


In [10]:
%%jshell

runPy("""
import pandas as pd

# Get the feature names from the dataset
feature_names = dataset.feature_names

# Fit PCA to the scaled data if it hasn't been fitted already
# This step calculates the principal components based on the scaled data.
pca.fit(scaled_data)

# Create a DataFrame to better visualize the principal components' loadings
# Each row is a principal component, and each column is an original feature
components_df = pd.DataFrame(pca.components_, columns=feature_names, index=[f'Principal Component {i+1}' for i in range(n_components)])

print("Principal Components Loadings:")
display(components_df)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Principal Components Loadings:
                       mean radius  ...  worst fractal dimension
Principal Component 1     0.218902  ...                 0.131784
Principal Component 2    -0.233857  ...                 0.275339

[2 rows x 30 columns]


The table above shows the loadings (coefficients) for each original feature on Principal Component 1 and Principal Component 2. A higher absolute value for a coefficient indicates that the corresponding original feature has a stronger influence on that principal component.

For example, if 'mean radius' has a high positive loading on Principal Component 1, it means that PC1 increases as 'mean radius' increases. If it has a high negative loading, PC1 decreases as 'mean radius' increases.

By examining these values, you can infer which original features are most strongly associated with each principal component and, consequently, which features contribute most to the variance captured by that component.

In [11]:
%%jshell

runPy("""
# Fit PCA to the scaled data
#
# This step calculates the principal components based on the scaled data.
pca.fit(scaled_data)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... PCA(n_components=2)


In [12]:
%%jshell

runPy("""
# Transform the data to its principal components
#
# This step applies the calculated principal components to the scaled data to reduce its dimensionality.
transformed_data = pca.transform(scaled_data)
transformed_data.shape
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... (569, 2)


In [13]:
%%jshell

runPy("""
# Explained variance ratio
# The explained variance ratio indicates the proportion of the dataset's variance that lies along each principal component.
explained_variance_ratio = pca.explained_variance_ratio_
print("Explained Variance Ratio:", explained_variance_ratio)

# Principal components
# The principal components are the directions in the original feature space along which the variance is maximized.
principal_components = pca.components_
print("Principal Components:", principal_components)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Explained Variance Ratio: [0.44272026 0.18971182]
Principal Components: [[ 0.21890244  0.10372458  0.22753729  0.22099499  0.14258969  0.23928535
   0.25840048  0.26085376  0.13816696  0.06436335  0.20597878  0.01742803
   0.21132592  0.20286964  0.01453145  0.17039345  0.15358979  0.1834174
   0.04249842  0.10256832  0.22799663  0.10446933  0.23663968  0.22487053
   0.12795256  0.21009588  0.22876753  0.25088597  0.12290456  0.13178394]
 [-0.23385713 -0.05970609 -0.21518136 -0.23107671  0.18611302  0.15189161
   0.06016536 -0.0347675   0.19034877  0.36657547 -0.10555215  0.08997968
  -0.08945723 -0.15229263  0.20443045  0.2327159   0.19720728  0.13032156
   0.183848    0.28009203 -0.21986638 -0.0454673  -0.19987843 -0.21935186
   0.17230435  0.14359317  0.09796411 -0.00825724  0.14188335  0.27533947]]


In [14]:
%%jshell

runPy("""
# Load the target variable for coloring the plot
target = dataset.target

# Create a scatter plot of the transformed data
plt.figure(figsize=(8, 6))
plt.scatter(transformed_data[:, 0], transformed_data[:, 1], c=target, cmap='viridis')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Breast Cancer Dataset (2 Components)')
plt.colorbar(label='Target (0: Malignant, 1: Benign)')
plt.grid(True)
plt.savefig('/content/regression_plot.png')
""");


In [15]:
%%jshell

runPy("""
# Load the target variable for coloring the plot
target = dataset.target

# Create a scatter plot of the transformed data
plt.figure(figsize=(8, 6))

# Separate data by target class for legend
transformed_data_malignant = transformed_data[target == 0]
transformed_data_benign = transformed_data[target == 1]

# Plot each class separately
plt.scatter(transformed_data_malignant[:, 0], transformed_data_malignant[:, 1], c='purple', label='Malignant (0)', alpha=0.7)
plt.scatter(transformed_data_benign[:, 0], transformed_data_benign[:, 1], c='yellow', label='Benign (1)', alpha=0.7)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Breast Cancer Dataset (2 Components)')
plt.legend() # Add the legend
plt.grid(True)
plt.savefig('/content/regression_plot.png')
""");


In [16]:
%%jshell

runPy("""
# Final interpretation:

# The scatter plot shows a clear separation between the two target classes
# (Malignant and Benign) in the 2-dimensional space created by the first two
# principal components.
#
# This suggests that these two components capture a
# significant amount of the variance that differentiates the two classes,
# making it much easier to visualize the data and potentially build a
# classification model on this reduced-dimensional data.

# The explained variance ratio shows that the first principal component captures
# about 44.3% of the variance, and the second captures about 19.0%. Together,
# they capture approximately 63.3% of the total variance in the original
# 30-dimensional dataset.

# While this doesn't capture all the variance, the clear separation in the plot
# indicates that the most important information for distinguishing between the
# two classes is largely preserved in these two components.

# Conclusion: PCA has transformed the 30-dimensional data into a 2-dimensional
# representation where the classes are visually distinguishable, which is a
# meaningful outcome for understanding the data structure and potentially
# for downstream tasks like classification.
""");


In [17]:
%%jshell

runPy("""
import pandas as pd

# Create a DataFrame from the transformed data
# transformed_data and target are available from previous steps.
df = pd.DataFrame(data=transformed_data, columns=['Principal Component 1', 'Principal Component 2'])

# Add the target variable to the DataFrame
df['Target'] = target

df.to_excel('final_dataset.xlsx',index=False)
""");


In [18]:
%%jshell

runPy("""
import pandas as pd

# Create a DataFrame from the transformed data
df = pd.DataFrame(data=transformed_data, columns=['Principal Component 1', 'Principal Component 2'])

# Add the target variable to the DataFrame
df['Target'] = target

# Display the first few rows of the new DataFrame
display(df.head())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...    Principal Component 1  Principal Component 2  Target
0               9.192837               1.948583       0
1               2.387802              -3.768172       0
2               5.733896              -1.075174       0
3               7.122953              10.275589       0
4               3.935302              -1.948072       0


In [19]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-28 20:33:13
